# Quantum Speedup for Reversible Markov Chains — Lazy Birth-Death Chain
### Implementation and Verification via GQET

This notebook is a companion to `02_non_reversible_chain.ipynb`.  
It demonstrates the **reversible** case of the quantum sampling algorithm from  
[*Quantum speedup for nonreversible Markov chains* — Claudon, Piquemal & Monmarché, *Nature Communications* 2025](https://doi.org/10.1038/s41467-025-56171-0).

Because the chain is **reversible**, the flat discriminant $D$ is symmetric and  
has a leading *eigenvalue* (not singular value) equal to $1$.  
The algorithm therefore uses **GQET** (Generalized Quantum Eigenvalue Transform)  
rather than GQSVT, which simplifies the circuit: no Hermitisation ancilla is needed.

---

**Outline**

1. [Markov Chain Definition](#1-markov-chain-definition)  
2. [Discriminant & Spectral Gap](#2-discriminant--spectral-gap)  
3. [Symmetric PUE (SPUE) and Walk Operator](#3-symmetric-pue-spue-and-walk-operator)  
4. [Chebyshev Fast-Forwarding Polynomial](#4-chebyshev-fast-forwarding-polynomial)  
5. [Quantum Circuit (GQET)](#5-quantum-circuit-gqet)  
6. [Simulation & Results](#6-simulation--results)  
7. [Complexity: Classical vs. Quantum](#7-complexity-classical-vs-quantum)  
8. [Noise Analysis](#8-noise-analysis)

## 0 · Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
from collections import Counter

from numpy.polynomial import Chebyshev, Polynomial, chebyshev
from scipy.linalg import null_space, block_diag
from scipy.optimize import minimize
from scipy.signal import fftconvolve

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import UnitaryGate
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.providers.fake_provider import GenericBackendV2

# ── Matplotlib style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi":        120,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.35,
    "font.size":         11,
})

PALETTE = ["#aec97e", "#ebdf5e", "#ffb81d", "#73b7d9",
           "#af7ec9", "#f08080", "#4c72b0", "#dd8452"]

---
## 1 · Markov Chain Definition

We use the **4-state lazy birth-death chain** on $S = \{0, 1, 2, 3\}$:

$$P = \begin{pmatrix}
1/2 & 1/2 & 0 & 0 \\
1/4 & 1/2 & 1/4 & 0 \\
0 & 1/4 & 1/2 & 1/4 \\
0 & 0 & 1/2 & 1/2
\end{pmatrix}$$

This chain is **reversible** (satisfies detailed balance with respect to $\pi$),  
which is what allows us to use the simpler GQET path.

In [ ]:
DIM = 4
S_space = np.arange(DIM)

eye   = np.eye(DIM)
basis = [eye[:, i:i+1] for i in range(DIM)]   # column vectors

# ── Markov kernel ─────────────────────────────────────────────────────────────
P = np.array([
    [1/2, 1/2,   0,   0],
    [1/4, 1/2, 1/4,   0],
    [  0, 1/4, 1/2, 1/4],
    [  0,   0, 1/2, 1/2],
], dtype=float)

# ── Stationary distribution ───────────────────────────────────────────────────
pi = np.array([1/6, 1/3, 1/3, 1/6])
pi_state = np.sqrt(pi).reshape(-1, 1)   # coherent stationary state |π⟩

assert np.allclose(P.sum(axis=1), 1.0),  "P is not row-stochastic"
assert np.allclose(pi @ P, pi),          "π is not stationary"

# ── Reversibility (detailed balance) ─────────────────────────────────────────
for x in range(DIM):
    for y in range(DIM):
        assert np.allclose(pi[x]*P[x,y], pi[y]*P[y,x]),             f"Detailed balance fails at ({x},{y})"

print("✓ Markov kernel P:")
print(P)
print(f"\n✓ Stationary distribution π = {pi}")
print("✓ Detailed balance verified — chain is reversible")

---
## 2 · Discriminant & Spectral Gap

For a **reversible** chain the flat and curved discriminants coincide.  
The discriminant is

$$D(x,y) = \sqrt{\frac{\pi(x)}{\pi(y)}}\,P(x,y),$$

which is a **symmetric** matrix (because detailed balance holds).  
Its leading eigenvalue is $\lambda_1 = 1$, with eigenvector $|\pi\rangle$.  
The **spectral gap** $\delta = 1 - \lambda_2$ governs the polynomial degree:  
$$d = O\!\left(\delta^{-1/2}\log(1/\varepsilon)\right).$$

In [ ]:
# ── Discriminant ──────────────────────────────────────────────────────────────
D = np.array([[np.sqrt(pi[x]/pi[y])*P[x,y] for y in range(DIM)] for x in range(DIM)])

assert np.allclose(D, D.T), "D must be symmetric for a reversible chain"

# ── Eigendecomposition ────────────────────────────────────────────────────────
evals_raw, evecs_raw = np.linalg.eigh(D)   # eigh guarantees real eigenvalues
idx          = np.argsort(evals_raw)[::-1]  # descending order
evals        = evals_raw[idx]
evecs        = evecs_raw[:, idx]

leading_eval = evals[0]
leading_evec = evecs[:, 0]

assert np.isclose(leading_eval, 1.0), "Leading eigenvalue must be 1"
assert np.allclose(np.abs(leading_evec), np.abs(pi_state.flatten()), atol=1e-8),     "Leading eigenvector must be |π⟩"

# ── Spectral gap ──────────────────────────────────────────────────────────────
spectral_gap = leading_eval - evals[1]

print(f"Eigenvalues of D : {np.round(evals, 6)}")
print(f"Spectral gap  δ  : {spectral_gap:.4f}  (λ₂ = {evals[1]:.4f})")

---
## 3 · Symmetric PUE (SPUE) and Walk Operator

**Partial isometry** (state-preparation map):
$$\square_P = \sum_{x,y \in S}\sqrt{P(x,y)}\,|x\rangle|y\rangle\langle x|,
\qquad \square_P^\dagger\square_P = I.$$

**Swap operator**:
$$S = \sum_{x,y \in S}|x,y\rangle\langle y,x|.$$

The pair $(S,\square_P)$ is a **symmetric** PUE of $D$: $\square_P^\dagger S\,\square_P = D$.

**Qubitized walk operator** (unitary, needed for GQSP):
$$W = \bigl(2\square_P\square_P^\dagger - I\bigr)S.$$

In [ ]:
# ── Partial isometry ──────────────────────────────────────────────────────────
square = sum(
    np.sqrt(P[x,y]) * np.kron(basis[x], basis[y]) @ basis[x].T
    for x in range(DIM) for y in range(DIM)
)
assert np.allclose(square.T @ square, np.eye(DIM)), "square must be a partial isometry"

# ── Swap operator ─────────────────────────────────────────────────────────────
swap = sum(
    np.kron(basis[x], basis[y]) @ np.kron(basis[y].T, basis[x].T)
    for x in range(DIM) for y in range(DIM)
)

# ── SPUE verification ─────────────────────────────────────────────────────────
assert np.allclose(square.T @ swap @ square, D), "(square, swap) is not a SPUE of D"
print("✓ (square, swap) is a SPUE of D")

# ── Walk operator W ───────────────────────────────────────────────────────────
R_op = 2 * (square @ square.T) - np.eye(DIM**2)
W    = R_op @ swap
assert np.allclose(W.T @ W, np.eye(DIM**2)), "W must be unitary"
print("✓ Walk operator W is unitary")

### 3.1 · Alternative construction of W (as in the paper)

The walk operator can equivalently be written as

$$W = V\bigl(I \otimes (2|0\rangle\langle 0| - I)\bigr)V^\dagger S,$$

where $V = \bigoplus_x V_x$ and $V_x|0\rangle = \sum_y \sqrt{P(x,y)}\,|y\rangle$.

In [ ]:
def complete_vector_to_unitary(v, atol=1e-12):
    """Return a unitary whose first column is the normalised vector v."""
    v = np.asarray(v, dtype=complex).reshape(-1, 1)
    assert np.isclose(np.linalg.norm(v), 1.0, atol=atol), "v must be normalised"
    U = np.hstack([v, null_space(v.conj().T)])
    assert np.allclose(U.conj().T @ U, np.eye(v.shape[0]), atol=atol)
    return U


def build_controlled_unitary_extension(P, atol=1e-12):
    """
    Build V = ⊕_x V_x where V_x|0⟩ = Σ_y √P[x,y] |y⟩.
    Returns the full block-diagonal unitary V and the list of blocks.
    """
    dim  = P.shape[0]
    blks = [complete_vector_to_unitary(np.sqrt(P[x]).astype(complex)) for x in range(dim)]
    V    = block_diag(*blks)
    assert np.allclose(V.conj().T @ V, np.eye(dim**2), atol=atol), "V must be unitary"
    return V, blks


V, V_blocks = build_controlled_unitary_extension(P)

rot_0 = 2 * basis[0] @ basis[0].T - np.eye(DIM)   # 2|0><0| - I
W_alt = V @ np.kron(np.eye(DIM), rot_0) @ V.T @ swap

assert np.allclose(W, W_alt), "Alternative W construction must match"
print("✓ Alternative W construction verified")

### 3.2 · Endianness correction for Qiskit

Mathematics uses **big-endian** ordering $|x,y\rangle \mapsto \mathrm{dim}\cdot x + y$,  
while Qiskit uses **little-endian** $|x,y\rangle \mapsto x + \mathrm{dim}\cdot y$.  
We conjugate by the permutation matrix $\Pi$ before wrapping any operator as a gate.

In [ ]:
def math_to_qiskit_perm(dim):
    """
    Permutation matrix Π such that Π|x,y⟩_math = |x,y⟩_qiskit.
    math index: dim*x + y  →  qiskit index: x + dim*y
    """
    N    = dim * dim
    Perm = np.zeros((N, N), dtype=complex)
    for x in range(dim):
        for y in range(dim):
            Perm[x + dim*y, dim*x + y] = 1.0
    return Perm


perm     = math_to_qiskit_perm(DIM)
V_qiskit = perm @ V      @ perm.conj().T
W_qiskit = perm @ W_alt  @ perm.conj().T

assert np.allclose(V_qiskit.conj().T @ V_qiskit, np.eye(DIM**2)), "V_qiskit not unitary"
assert np.allclose(W_qiskit.conj().T @ W_qiskit, np.eye(DIM**2)), "W_qiskit not unitary"
print("✓ Qiskit-ordered V and W verified unitary")

---
## 4 · Chebyshev Fast-Forwarding Polynomial

We apply the polynomial
$$\nu(x) = \varepsilon\,T_d(\alpha x),
\quad \alpha = \cosh\!\left(\frac{\operatorname{arccosh}(1/\varepsilon)}{d}\right),$$
to the eigenvalues of $D$.  By construction $\nu(\lambda_1) = \nu(1) = 1$ and  
$|\nu(x)| \le \varepsilon$ for all other eigenvalues, so $\nu(D) \approx |\pi\rangle\langle\pi|$.

The minimum degree is
$$d = \left\lceil\frac{\operatorname{arccosh}(1/\varepsilon)}{\operatorname{arccosh}(1/(1-\delta))}\right\rceil.$$

In [ ]:
EPSILON = 0.125

d_raw = np.arccosh(1/EPSILON) / np.arccosh(1/(1-spectral_gap))
d     = int(np.ceil(d_raw))

alpha   = np.cosh(np.arccosh(1/EPSILON) / d)
delta_d = 1 - 1/alpha
assert delta_d <= spectral_gap + 1e-12, "Degree too small: delta_d > spectral_gap"

x_poly   = Polynomial([0.0, alpha])
nu_power = EPSILON * Chebyshev.basis(d)(x_poly)
nu       = nu_power.convert(kind=Chebyshev)
P_coefs  = np.array(nu.coef)

print(f"ε         = {EPSILON}")
print(f"d (raw)   = {d_raw:.4f}")
print(f"d (used)  = {d}")
print(f"α         = {alpha:.6f}")

In [ ]:
# ── Classical verification: nu(D) ≈ |π⟩⟨π| ──────────────────────────────────
def cheby_rec(deg, M):
    """Evaluate T_deg(M) via the three-term Chebyshev recurrence."""
    if deg == 0: return np.eye(M.shape[0])
    if deg == 1: return M.copy()
    Tm2, Tm1 = np.eye(M.shape[0]), M.copy()
    for _ in range(2, deg+1):
        T = 2*M@Tm1 - Tm2; Tm2, Tm1 = Tm1, T
    return Tm1

nu_D = sum(a * cheby_rec(n, D) for n, a in enumerate(P_coefs))

projector_pi = pi_state @ pi_state.T
filter_err   = np.linalg.norm(nu_D - projector_pi, ord=2)

assert filter_err <= EPSILON + 1e-10, "nu(D) not close enough to |π⟩⟨π|"
print(f"✓ ‖ν(D) − |π⟩⟨π|‖₂ = {filter_err:.5f}  (≤ ε = {EPSILON})")

filtered_evals = chebyshev.chebval(evals, P_coefs)
assert np.isclose(filtered_evals[0], 1.0),               "ν(λ₁) should be 1"
assert np.all(np.abs(filtered_evals[1:]) <= EPSILON+1e-10), "Non-leading evals exceed ε"
print(f"  Filtered eigenvalues: {np.round(filtered_evals, 5)}")

In [ ]:
# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: eigenvalue evolution
ax = axes[0]
for i in range(DIM):
    ax.plot(["Before filtering", "After filtering"],
            [evals[i], filtered_evals[i]],
            marker="o", ms=8, lw=2.2,
            color=PALETTE[i], label=f"λ_{i+1}")
ax.axhline(EPSILON,  color="crimson", ls="--", lw=1.5, label=f"ε = {EPSILON}")
ax.axhline(-EPSILON, color="crimson", ls="--", lw=1.5, alpha=0.5)
ax.set_title("Eigenvalues before and after filtering")
ax.set_ylabel("Value"); ax.legend(fontsize=9)

# Right: polynomial shape
ax = axes[1]
x_plt = np.linspace(-1.5, 1.5, 1000)
ax.plot(x_plt, nu(x_plt), color=PALETTE[0], lw=2.2,
        label=f"ν(x) = ε·T_{d}(αx)")
ax.axhline( EPSILON, color="crimson", ls="--", lw=1.5, label=f"±ε = {EPSILON}")
ax.axhline(-EPSILON, color="crimson", ls="--", lw=1.5, alpha=0.5)
ax.axvline( 1/alpha, color="steelblue", ls=":", lw=1.5, label="±1/α threshold")
ax.axvline(-1/alpha, color="steelblue", ls=":", lw=1.5)
ax.fill_between(x_plt, -EPSILON, EPSILON,
                where=np.abs(x_plt) <= 1/alpha, color="steelblue", alpha=0.08)
ax.scatter(evals, nu(evals), color="crimson", zorder=5, label="Eigenvalues")
ax.set_ylim(-0.4, 1.4); ax.set_xlabel("x"); ax.set_ylabel("ν(x)")
ax.set_title("Chebyshev filter polynomial"); ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 5 · Quantum Circuit (GQET)

We implement
$$\bigl(\langle 0|\otimes\square_P^\dagger\bigr)\,G\,\bigl(|0\rangle\otimes\square_P\bigr)
\;=\; \nu(D) \;\approx\; |\pi\rangle\langle\pi|,$$
where $G$ is the GQSP circuit that applies $\nu$ to the walk operator $W$.

**Qubit registers**

| Register | Size | Role |
|---|---|---|
| `s` | 1 | GQSP signal ancilla |
| `x` | 2 | current state ($2^2 = 4$ states) |
| `y` | 2 | transition register |

**Post-selection condition:** `s = 0`.

### 5.1 · GQSP helper functions

In [ ]:
def find_Q_polynomial(P_coefs):
    """
    Find Q such that |P(z)|² + |Q(z)|² = 1 on the unit circle.
    Solves the optimisation problem from Motlagh & Wiebe (2024) via L-BFGS-B.

    Args:
        P_coefs: Signal polynomial coefficients [a_0, ..., a_d] (real for our case).

    Returns:
        Q_coefs: Complex coefficients [b_0, ..., b_d].
    """
    a   = np.array(P_coefs, dtype=complex)
    d   = len(a) - 1
    rhs = np.zeros(2*d+1, dtype=complex)
    rhs[d] = 1.0
    rhs -= fftconvolve(a, np.conj(a[::-1]))

    def objective(b_flat):
        b = b_flat[:d+1] + 1j*b_flat[d+1:]
        return np.sum(np.abs(fftconvolve(b, np.conj(b[::-1])) - rhs)**2)

    np.random.seed(42)
    b_init = np.random.randn(d+1) + 1j*np.random.randn(d+1)
    x0     = np.concatenate([np.real(b_init), np.imag(b_init)])

    res = minimize(objective, x0, method="L-BFGS-B",
                   options={"maxiter": 50000, "ftol": 1e-12})
    if not res.success:
        print(f"  ⚠ Q optimisation: {res.message}  (residual {res.fun:.2e})")

    return res.x[:d+1] + 1j*res.x[d+1:]


def compute_rotation_angles(S, d):
    """
    Compute GQSP rotation angles from the stacked polynomial matrix S.

    Args:
        S: (2, d+1) complex array — first row P coefficients, second row Q coefficients.
        d: Polynomial degree.

    Returns:
        theta_list, phi_list  (each length d+1, ordered 0 → d),
        lambda_val            (global phase).
    """
    thetas, phis = [], []
    cur = np.array(S, dtype=complex)

    for k in range(d, 0, -1):
        a_k, b_k  = cur[0, k], cur[1, k]
        abs_a, abs_b = np.abs(a_k), np.abs(b_k)
        theta_k = np.arctan2(abs_b, abs_a)
        phi_k   = (np.angle(a_k * np.conj(b_k)) if (abs_a > 0 and abs_b > 0)
                   else (np.angle(a_k) if abs_b == 0 else -np.angle(b_k)))
        thetas.append(theta_k); phis.append(phi_k)

        R_dag = np.array([
            [np.exp(-1j*phi_k)*np.cos(theta_k),  np.sin(theta_k)],
            [np.exp(-1j*phi_k)*np.sin(theta_k), -np.cos(theta_k)],
        ])
        nxt  = R_dag @ cur
        cur  = np.vstack([nxt[0, 1:], nxt[1, :-1]])

    a0, b0    = cur[0, 0], cur[1, 0]
    abs_a0, abs_b0 = np.abs(a0), np.abs(b0)
    theta_0 = np.arctan2(abs_b0, abs_a0)
    phi_0   = (np.angle(a0*np.conj(b0)) if (abs_a0>0 and abs_b0>0)
               else (np.angle(a0) if abs_b0==0 else -np.angle(b0)))
    lam     = np.angle(b0) if abs_b0 > 0 else 0.0

    thetas.append(theta_0); phis.append(phi_0)
    return list(reversed(thetas)), list(reversed(phis)), lam


def rotation_gate(theta, phi, lam):
    """GQSP rotation R(θ, φ, λ) as a UnitaryGate."""
    U = np.array([
        [np.exp(1j*(lam+phi))*np.cos(theta),  np.exp(1j*phi)*np.sin(theta)],
        [np.exp(1j*lam)      *np.sin(theta), -np.cos(theta)],
    ], dtype=complex)
    return UnitaryGate(U, label=r"$R_{\mathrm{GQSP}}$")

print("✓ GQSP helper functions defined")

### 5.2 · Compute GQSP angles

In [ ]:
print(f"Computing GQSP angles for d = {d} ...")

Q_coefs = find_Q_polynomial(P_coefs)
S_mat   = np.vstack([P_coefs, Q_coefs])
theta_list, phi_list, lambda_val = compute_rotation_angles(S_mat, d)

print(f"✓ Computed {len(theta_list)} rotation angles")
print(f"  λ = {lambda_val:.6f}")

### 5.3 · Build the quantum circuit

In [ ]:
# ── Gates ─────────────────────────────────────────────────────────────────────
V_gate  = UnitaryGate(V_qiskit,           label="V")
Vd_gate = UnitaryGate(V_qiskit.conj().T,  label=r"V†")
W_gate  = UnitaryGate(W_qiskit,           label="W")
A_gate  = W_gate.control(num_ctrl_qubits=1, ctrl_state="0")  # controlled-W on |0⟩

# ── Registers ─────────────────────────────────────────────────────────────────
qs = QuantumRegister(1, "s")
qx = QuantumRegister(2, "x")
qy = QuantumRegister(2, "y")
cs = ClassicalRegister(1, "cs")
cx = ClassicalRegister(2, "cx")

qc = QuantumCircuit(qs, qx, qy, cs, cx)

# ── Initialisation: equal superposition over x ────────────────────────────────
for q in qx:
    qc.h(q)
qc.barrier()

# ── Apply V ───────────────────────────────────────────────────────────────────
qc.append(V_gate, list(qx) + list(qy))
qc.barrier()

# ── GQSP loop ─────────────────────────────────────────────────────────────────
for j, theta_j in enumerate(theta_list):
    if j > 0:
        qc.append(A_gate, list(qs) + list(qx) + list(qy))
    lam_j = lambda_val if j == 0 else 0.0
    qc.append(rotation_gate(theta_j, phi_list[j], lam_j), qs)
qc.barrier()

# ── Apply V† ─────────────────────────────────────────────────────────────────
qc.append(Vd_gate, list(qx) + list(qy))
qc.barrier()

# ── Measurements ──────────────────────────────────────────────────────────────
qc.measure(qs, cs)
qc.measure(qx, cx)

print(f"✓ Circuit built  (depth = {qc.depth()}, qubits = {qc.num_qubits})")
qc.draw("mpl", fold=50)

---
## 6 · Simulation & Results

We run on a **noiseless** Qiskit Aer simulator and post-select on `s = 0`.

In [ ]:
SHOTS = 10_000
ZERO_Y = "00"

simulator = AerSimulator()
raw_counts = simulator.run(transpile(qc, simulator), shots=SHOTS).result().get_counts()

# ── Post-selection ────────────────────────────────────────────────────────────
post_counts = {}
n_success   = 0

for bs, cnt in raw_counts.items():
    cx_val, cs_val = bs.split()
    if cs_val == "0":
        post_counts[cx_val] = post_counts.get(cx_val, 0) + cnt
        n_success += cnt

success_rate = n_success / SHOTS
measured     = np.array([post_counts.get(f"{i:02b}", 0) / n_success for i in range(DIM)])
tvd          = 0.5 * np.sum(np.abs(pi - measured))

print(f"Post-selection success rate : {success_rate*100:.2f}%")
print(f"TVD(π̃, π)                  : {tvd:.5f}")
print()
print(f"{'State':>6}  {'π̃ (measured)':>14}  {'π (theory)':>12}  {'|Δ|':>8}")
print("-"*46)
for i in range(DIM):
    print(f"  |{i}⟩   {measured[i]:14.6f}  {pi[i]:12.6f}  {abs(measured[i]-pi[i]):8.6f}")

In [ ]:
# ── Bar chart ─────────────────────────────────────────────────────────────────
states = np.arange(DIM); w = 0.38
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(states - w/2, measured, w, label="Measured π̃", color=PALETTE[0], alpha=0.85)
ax.bar(states + w/2, pi,       w, label="Theory π",   color=PALETTE[3], alpha=0.85)
ax.set_xticks(states, [f"|{i}⟩" for i in states])
ax.set_xlabel("State"); ax.set_ylabel("Probability")
ax.set_title(f"Stationary distribution (GQET, d={d}, TVD={tvd:.4f})")
ax.legend(); plt.tight_layout(); plt.show()

---
## 7 · Complexity: Classical vs. Quantum

For each degree $d$ we compare:
- **Classical**: $d$ applications of $P$ from the uniform distribution.
- **Quantum**: GQET circuit with Chebyshev polynomial of degree $d$.

The quantum TVD should fall much faster than the classical one,  
demonstrating the quadratic speedup.

In [ ]:
def classic_tvd(degree, P, pi, start=None):
    """TVD after `degree` steps of P from `start` (default: uniform)."""
    dist = np.ones(DIM)/DIM if start is None else start.copy()
    for _ in range(degree):
        dist = dist @ P
    return 0.5 * np.sum(np.abs(dist - pi))


def quantum_tvd(P, pi, epsilon=EPSILON, degree=None, shots=SHOTS, simulator=None):
    """
    Build and simulate the GQET circuit for reversible kernel P.

    Returns:
        measured: empirical probability vector
        tvd:      Total Variation Distance from π
    """
    if simulator is None:
        simulator = AerSimulator()

    dim = P.shape[0]; n_q = int(np.log2(dim))
    assert 2**n_q == dim

    eye_  = np.eye(dim); bas_ = [eye_[:,i:i+1] for i in range(dim)]
    pi_   = pi / pi.sum()

    # Discriminant and spectral gap
    D_    = np.array([[np.sqrt(pi_[x]/pi_[y])*P[x,y] for y in range(dim)] for x in range(dim)])
    ev_,_ = np.linalg.eigh(D_)
    gap_  = 1.0 - sorted(ev_)[-2]

    # Operators
    sq_   = sum(np.sqrt(P[x,y])*np.kron(bas_[x],bas_[y])@bas_[x].T for x in range(dim) for y in range(dim))
    sw_   = sum(np.kron(bas_[x],bas_[y])@np.kron(bas_[y].T,bas_[x].T) for x in range(dim) for y in range(dim))
    rot0_ = 2*bas_[0]@bas_[0].T - np.eye(dim)
    V_,_  = build_controlled_unitary_extension(P)
    W_    = V_ @ np.kron(np.eye(dim), rot0_) @ V_.T @ sw_

    pm_   = math_to_qiskit_perm(dim)
    Vq_   = pm_ @ V_ @ pm_.conj().T
    Wq_   = pm_ @ W_ @ pm_.conj().T

    # Polynomial
    d_ = degree if degree is not None else int(np.ceil(
        np.arccosh(1/epsilon) / np.arccosh(1/(1-gap_))))
    al_ = np.cosh(np.arccosh(1/epsilon) / d_)
    nu_ = (epsilon * Chebyshev.basis(d_)(Polynomial([0.0, al_]))).convert(kind=Chebyshev)
    Pc_ = np.array(nu_.coef)
    Qc_ = find_Q_polynomial(Pc_)
    th_, ph_, lv_ = compute_rotation_angles(np.vstack([Pc_, Qc_]), d_)

    # Circuit
    qs_=QuantumRegister(1,"s"); qx_=QuantumRegister(n_q,"x"); qy_=QuantumRegister(n_q,"y")
    cs_=ClassicalRegister(1,"cs"); cx_=ClassicalRegister(n_q,"cx")
    qc_=QuantumCircuit(qs_,qx_,qy_,cs_,cx_)
    for q in qx_: qc_.h(q)
    qc_.append(UnitaryGate(Vq_,label="V"), list(qx_)+list(qy_))
    Ag_ = UnitaryGate(Wq_,label="W").control(1,ctrl_state="0")
    for j,th in enumerate(th_):
        if j>0: qc_.append(Ag_, list(qs_)+list(qx_)+list(qy_))
        lj = lv_ if j==0 else 0.0
        qc_.append(UnitaryGate(np.array([
            [np.exp(1j*(lj+ph_[j]))*np.cos(th),  np.exp(1j*ph_[j])*np.sin(th)],
            [np.exp(1j*lj)         *np.sin(th), -np.cos(th)],
        ],dtype=complex)), qs_)
    qc_.append(UnitaryGate(Vq_.conj().T,label="V†"), list(qx_)+list(qy_))
    qc_.measure(qs_,cs_); qc_.measure(qx_,cx_)

    raw_ = simulator.run(transpile(qc_,simulator), shots=shots).result().get_counts()
    pc_={}; ns_=0
    for bs,cnt in raw_.items():
        cv,sv = bs.split()
        if sv=="0": pc_[cv]=pc_.get(cv,0)+cnt; ns_+=cnt

    if ns_==0: return np.zeros(dim), 1.0
    meas_ = np.array([pc_.get(f"{i:0{n_q}b}",0)/ns_ for i in range(dim)])
    return meas_, 0.5*np.sum(np.abs(pi_-meas_))

print("✓ Complexity analysis functions defined")

In [ ]:
from IPython.display import clear_output

degree_vals     = np.arange(1, 10)
tvd_classic_lst = []
tvd_quantum_lst = []

for deg in degree_vals:
    tvd_c = classic_tvd(deg, P, pi)
    _, tvd_q = quantum_tvd(P, pi, degree=deg)
    tvd_classic_lst.append(tvd_c)
    tvd_quantum_lst.append(tvd_q)

    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(degree_vals[:len(tvd_classic_lst)], tvd_classic_lst,
            "o-", color=PALETTE[0], lw=2, ms=5, label="Classical")
    ax.plot(degree_vals[:len(tvd_quantum_lst)], tvd_quantum_lst,
            "o-", color=PALETTE[4], lw=2, ms=5, label="Quantum (GQET)")
    ax.set_xlabel("Degree $d$"); ax.set_ylabel("TVD")
    ax.set_title("TVD vs. Degree — Classical vs. Quantum"); ax.legend()
    ax.set_xticks(degree_vals[:len(tvd_classic_lst)])
    plt.tight_layout(); plt.show()
    print(f"  d={deg}  classical={tvd_c:.5f}  quantum={tvd_q:.5f}")

In [ ]:
# ── Sweep epsilon ─────────────────────────────────────────────────────────────
from IPython.display import clear_output

eps_vals    = np.arange(0.01, 0.50, 0.01)
tvd_eps_lst = []

for eps in eps_vals:
    _, tvd_q = quantum_tvd(P, pi, epsilon=eps)
    tvd_eps_lst.append(tvd_q)

    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(eps_vals[:len(tvd_eps_lst)], tvd_eps_lst,
            color=PALETTE[0], lw=2.2)
    ax.set_xlabel("ε"); ax.set_ylabel("TVD")
    ax.set_title("Effect of ε on TVD"); plt.tight_layout(); plt.show()
    print(f"  ε={eps:.2f}  quantum TVD={tvd_q:.5f}")

---
## 8 · Noise Analysis

We study the effect of **depolarising noise** on the output TVD,  
sweeping the 2-qubit error rate $p$ (1-qubit rate = $p/10$).

In [ ]:
def make_noise_model(p):
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(p/10, 1), ["x","sx","rz"])
    nm.add_all_qubit_quantum_error(depolarizing_error(p,    2), ["cx"])
    return nm

p_vals   = np.arange(0, 5.1e-3, 2e-4)
tvd_noise = []

for p in p_vals:
    sim_n = AerSimulator(noise_model=make_noise_model(p))
    _, tvd_n = quantum_tvd(P, pi, simulator=sim_n)
    tvd_noise.append(tvd_n)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(p_vals*1e3, tvd_noise, color=PALETTE[0], lw=2.2, marker="o", ms=4)
ax.axhline(EPSILON, color="crimson", ls="--", lw=1.5, label=f"ε = {EPSILON}")
ax.set_xlabel("2-qubit depolarising error rate $p$ (×10⁻³)")
ax.set_ylabel("TVD(π̃, π)")
ax.set_title("Effect of depolarising noise on TVD")
ax.legend(); plt.tight_layout(); plt.show()